# Day28 Hybrid GPU Notebook (Colab)
Run vLLM on Colab GPU and copy public URL to local `.env`.

In [ ]:
!pip -q install vllm pyngrok requests

In [ ]:
from pyngrok import ngrok
NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN"
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()
print('ngrok ready')

In [ ]:
import subprocess, threading

def run_vllm():
    subprocess.run([
        "python","-m","vllm.entrypoints.openai.api_server",
        "--model","Qwen/Qwen2.5-0.5B-Instruct",
        "--host","0.0.0.0",
        "--port","8001",
        "--max-model-len","1024",
        "--gpu-memory-utilization","0.35",
        "--dtype","float16",
        "--enforce-eager"
    ])

threading.Thread(target=run_vllm, daemon=True).start()
print('vLLM starting...')

In [ ]:
import requests, time
ok = False
for i in range(40):
    try:
        r = requests.get('http://localhost:8001/v1/models', timeout=10)
        print(i*5, r.status_code)
        if r.status_code == 200:
            ok = True
            break
    except Exception:
        pass
    time.sleep(5)

if not ok:
    raise RuntimeError('vLLM not healthy yet')

In [ ]:
vllm_tunnel = ngrok.connect(8001, 'http')
VLLM_NGROK_URL = vllm_tunnel.public_url
print('VLLM_NGROK_URL=', VLLM_NGROK_URL)

import requests
print(requests.get(f'{VLLM_NGROK_URL}/v1/models', timeout=30).status_code)

Copy URL into local `.env`:

```env
VLLM_NGROK_URL=https://...
```